# Shared Setup

This notebook is the common starting point for the group.

## Goals
- load the Kaggle dataset
- check class balance
- build one preprocessing flow for everyone

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import RAW_DATA_DIR
from src.preprocessing import make_shared_preprocessing_pipeline, split_data

sns.set_theme(style="whitegrid")
print("Ready to load data from:", RAW_DATA_DIR)

In [ ]:
csv_files = sorted(RAW_DATA_DIR.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError('No CSV file found in data/raw. Put the Kaggle CSV there first.')

dataset_path = csv_files[0]
df = pd.read_csv(dataset_path)
print('Loaded:', dataset_path.name)
print('Shape:', df.shape)
display(df.head())

In [ ]:
print('Missing values:')
display(df.isna().sum().sort_values(ascending=False).head(10))

print('Column names:')
print(list(df.columns))

target_column = 'Machine failure' if 'Machine failure' in df.columns else None
if target_column is not None:
    print('Target balance:')
    display(df[target_column].value_counts(normalize=True).rename('ratio'))
    ax = df[target_column].value_counts().plot(kind='bar', title='Target distribution')
    ax.set_xlabel('Machine failure')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No target column named Machine failure was found yet.')

In [ ]:
shared = make_shared_preprocessing_pipeline(df)
X_train, X_test, y_train, y_test = split_data(df, target=shared['target_column'])

print('Target column:', shared['target_column'])
print('Dropped columns:', shared['dropped_columns'])
print('Train shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)
print('Numeric features:', X_train.select_dtypes(include=['number']).columns.tolist())
print('Categorical features:', X_train.select_dtypes(exclude=['number']).columns.tolist())

X_train_ready = shared['preprocessor'].fit_transform(X_train)
X_test_ready = shared['preprocessor'].transform(X_test)
print('Processed train shape:', X_train_ready.shape)
print('Processed test shape:', X_test_ready.shape)